In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import json

def scrape(path):
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(path)
    driver.maximize_window()


    auction_data = {}
    try:
        auction_name = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/div[@class="auction-header row"]/h1'))
        ).text.strip()
        auction_data["auctionname"] = auction_name
    except:
        auction_data["auctionname"] = ""

    try:

        auction_ul = driver.find_element(By.XPATH, './/div[@class="auction-header row"]/ul')
        lis = auction_ul.find_elements(By.TAG_NAME, "li")
        auction_data["auctioncenter"] = lis[0].text.strip() if len(lis) > 0 else ""
        auction_data["auctiontime"] = lis[1].text.strip() if len(lis) > 1 else ""
    except:
        auction_data["auctioncenter"] = ""
        auction_data["auctiontime"] = ""


    if not os.path.exists("database"):
        os.makedirs("database")
    with open("database/db.json", "w") as f:
        json.dump(auction_data, f, indent=4)
    print("Auction details saved to database/db.json")


    try:
        cars_strong = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/span[@class="sort-page"]/strong'))
        )
        total_cars = int(cars_strong.text.strip())
        print(f"{total_cars} cars found")
    except Exception as e:
        print("Cannot read total cars:", e)
        total_cars = 0

    car_count = 0
    while car_count < total_cars:
        try:
            page_cars = WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located((By.XPATH, './/img[@class="card-img-top"]'))
            )
            for i in range(len(page_cars)):
                if car_count >= total_cars:
                    break

                page_cars = WebDriverWait(driver, 5).until(
                    EC.presence_of_all_elements_located((By.XPATH, './/img[@class="card-img-top"]'))
                )
                driver.execute_script("arguments[0].scrollIntoView();", page_cars[i])
                WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable(page_cars[i])
                ).click()

      
                try:
                    # Get registration number from span.pill-item-reg
                    reg_el = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.XPATH, './/span[contains(@class,"pill-item-reg")]'))
                    )
                    reg_number = reg_el.text.strip()
                    # Sanitize filename (remove forbidden characters just in case)
                    invalid_chars = '<>:"/\\|?*'
                    for ch in invalid_chars:
                        reg_number = reg_number.replace(ch, "_")

                    if not os.path.exists("html"):
                        os.makedirs("html")
                    filename = f"html/{reg_number}.html"
                    with open(filename, "w", encoding="utf-8") as f:
                        f.write(driver.page_source)
                    print(f"✔ Saved HTML: {filename}")
                except Exception as e:
                    print("HTML save error:", e)


                car_count += 1
                driver.back()
                WebDriverWait(driver, 5).until(
                    EC.presence_of_all_elements_located((By.XPATH, './/img[@class="card-img-top"]'))
                )

       
            try:
                next_btn = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.XPATH, '//li[@class="page-item page-item-arrow page-item-arrow-next"]/a'))
                )
                next_href = next_btn.get_attribute("href")
                if next_href:
                    print(f"➡ Next page: {next_href}")
                    driver.get(next_href)
                    time.sleep(1)  
                else:
                    print("No more pages")
                    break
            except:
                print("Pagination finished")
                break
        except:
            print("No more car links")
            break

    driver.quit()
    print("Scraping completed!")


path = 'https://www.leominstercarauctions.co.uk/auction/278'
scrape(path)


Auction details saved to database/db.json
30 cars found
✔ Saved HTML: html/DX15YJM.html
✔ Saved HTML: html/EJ12AZN.html
✔ Saved HTML: html/HG56XBS.html
✔ Saved HTML: html/CP59NEY.html
✔ Saved HTML: html/DA62JPF.html
✔ Saved HTML: html/OU53LCE.html
✔ Saved HTML: html/BW11OMF.html
✔ Saved HTML: html/VO61FKN.html
✔ Saved HTML: html/LY61MKZ.html
✔ Saved HTML: html/GU54ESG.html
✔ Saved HTML: html/FG13FUP.html
✔ Saved HTML: html/SJ63EZS.html
✔ Saved HTML: html/WL57BWU.html
✔ Saved HTML: html/MF11XCE.html
✔ Saved HTML: html/YY64MYM.html
✔ Saved HTML: html/HN61EHC.html
✔ Saved HTML: html/OE10XEK.html
✔ Saved HTML: html/YC55SWW.html
✔ Saved HTML: html/CX58MXT.html
✔ Saved HTML: html/AF06MYX.html
✔ Saved HTML: html/DA66JVV.html
✔ Saved HTML: html/SG59KZR.html
✔ Saved HTML: html/WJ59FRV.html
✔ Saved HTML: html/CE12KYC.html
✔ Saved HTML: html/AJ15OCY.html
✔ Saved HTML: html/AK10LLV.html
✔ Saved HTML: html/RE16NFT.html
✔ Saved HTML: html/CN15XCU.html
✔ Saved HTML: html/DS62TVX.html
✔ Saved HTML: ht

In [31]:
import os,re,json
import csv
import datetime
from bs4 import BeautifulSoup

def fecthDetails(soup):    
    output = {}

    if soup:
        mainDiv = soup.find("ul", class_="details-list")
        if mainDiv:
            items = mainDiv.find_all("li", class_="detail-item")

            for item in items:
                key_tag = item.find("span")
                value_tag = item.find("strong")

                if key_tag and value_tag:
                    key = key_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    output[key] = value

    return output
    
    
def extract_image_urls(soup):
    image_urls = []

    if not soup:
        return ""

    thumbs_div = soup.find("div", class_="ug-thumbs-strip")
    if thumbs_div:
        imgs = thumbs_div.find_all("img", class_="ug-thumb-image")

        for img in imgs:
            src = img.get("src", "")
            if not src:
                continue

            # Remove API prefix
            src = src.replace(
                "https://dgaww6lqj3.execute-api.eu-west-1.amazonaws.com/prod/buckets/leominster-buckets-s3-public/keys/",
                ""
            )

            # Remove resized folder
            src = src.replace("/resized/", "/")

            # Remove ---177-100, ---150-100 etc.
            # Pattern: '---{width}-{height}'
            if "---" in src:
                src = src.split("---")[0] + ".jpg"

            # Final S3 URL
            new_url = f"https://leominster-buckets-s3-public.s3.eu-west-1.amazonaws.com/{src}"

            image_urls.append(new_url)

    return ",".join(image_urls)
 
    
    
def extract_manual_keys():
    folder = "html"
    output_file = "leominstercarauctions_data.csv"

    keys = ["Title",
            "Auction Name",
            "Lot", 
            "Auction type",
            "Center",
            "Make",
            "Model",
            "Variant",
            "Doors",
            "Reg", 
            "Start Time", 
            "Start Date",
            "D.O.R",
            "Fuel Type",
            "Body Type",
            "Former Keepers",
            "Transmission",
            "Colour",
            "MOT Expiry Date",
            "Year",
            "Keys",
            "VAT Status",
            "V5",
            "CAP Clean",
            "CAP Average",
            "CAP Below",
            "CC",
            "Mileage",
            "Mileage Warranted",
            # "Additional information",
            # "General Condition",
            # "Tyres Condition",
            # "Euro Status",
            # "MOT Due",
            # "Inspection Report",
            "Images",
            # "Damaged_images",
            # "Damage_details",
            ]  

    all_rows = []

    for file in os.listdir(folder):
        if file.endswith(".html"):
            file_path = os.path.join(folder, file)
            with open(file_path, "r", encoding="utf-8") as f:
                html_content = f.read()
                soup = BeautifulSoup(html_content, "html.parser")

            row = {}
           
            title_tag = soup.find("h1",class_="title-h1")
            if title_tag:
                title_text = title_tag.get_text(strip=True)
                row["Title"] = title_text
                match_object = re.match(r"(\S+)\s+(.+)",title_text)
                if match_object:
                    row["Make"]=match_object.group(1)
                    row["Model"]=match_object.group(2)
                else:
                    row["Make"]=""
                    row["Model"]=""
                    
                
            else:
                row["Title"] = ""

            with open("database/db.json") as datab:
                db = json.load(datab)
            row['Auction Name'] = db.get("auctionname","") 
            
            lot_tag = soup.find("span" ,class_="pill-item pill-item-lot")
            if lot_tag:
                lot_text = lot_tag.get_text(strip=True).replace("Lot","")
                row["Lot"] = lot_text
            else:
                row["Lot"] = ""
            reg_tag = soup.find("span" ,class_="pill-item pill-item-reg")
            if reg_tag:
                Reg_text = reg_tag.get_text(strip=True)
                row["Reg"] = Reg_text
            else:
                row["Reg"] = ""
            row['Auction type'] = "Online Auction"
            row['Center'] = "Leominster"
            
            subs=soup.find("p",class_="title-sub title-sub-2")
            variant_full = subs.get_text(strip=True) 
            doors_match = re.search(r"\b(\d+)dr\b", variant_full.lower())
            doors = doors_match.group(1) if doors_match else ""
            variant_full = re.sub(r"\b\d+dr\b", "", variant_full, flags=re.IGNORECASE).strip()
            cc_match = re.search(r"\b(\d+\.\d+|\d{3,4})\b", variant_full)
            cc = cc_match.group(1) if cc_match else ""
            if cc:
                variant_full = re.sub(cc, "", variant_full).strip()
            variant_cleaned = re.sub(r"\s+", " ", variant_full).strip()
            
            row["Variant"] = variant_cleaned
            row["Doors"] = doors
            row["CC"] = cc
            timeandDate_tag = soup.find("span",class_="pill-item pill-item-time")
            timeandDate=timeandDate_tag.get_text(strip=True)
            if timeandDate:
         
                parts = timeandDate.split("-")
                time_part = parts[0].strip()     
                date_part = parts[1].strip()    


                time_24 = datetime.datetime.strptime(time_part, "%I%p").strftime("%H:%M")

                row["Start Time"] = time_24
                row["Start Date"] = date_part
            else:
                row["Start Time"] = ""
                row["Start Date"] = ""
            get_details = fecthDetails(soup)
            row["D.O.R"] = get_details.get("Registered")
            row["CAP Clean"] = get_details.get("CAP Clean")
            row["CAP Average"] = get_details.get("CAP Average")
            row["CAP Below"] = get_details.get("CAP Below")
            row["Fuel Type"] = get_details.get("Fuel")
            row["Body Type"] = get_details.get("Body type")
            row["V5"] = get_details.get("V5")
            row["VAT Status"] = get_details.get("VAT")
            row["Keys"] = get_details.get("Keys")
            row["Transmission"] = get_details.get("Transmission")
            row["Colour"] = get_details.get("Colour")
            row["Former Keepers"] = get_details.get("Former Keepers")
            row["MOT Expiry Date"] = get_details.get("MOT Expires")
            yearCovert = get_details.get("Registered")
            yearSplite = yearCovert.split("/")
            year = yearSplite[-1] 
            row["Year"] = year
            
            mil = get_details.get("Mileage", "")
            milage = ""
            if mil:
                milage = "".join(filter(str.isdigit, mil))
            row["Mileage"] = milage
            row["Mileage Warranted"] = "Yes" if get_details.get("Warranted","") == "Warranted" else"No"
            images= extract_image_urls(soup)
            row['Images'] = images if images is not None else ""
            
            
            all_rows.append(row)
           


    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=keys)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✔ CSV Generated: {output_file}")


extract_manual_keys()



✔ CSV Generated: leominstercarauctions_data.csv


In [30]:
from urllib.parse import urlparse, urljoin
import threading, requests, os, re
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

df = pd.read_csv("leominstercarauctions_data.csv")

reg_img = df[['Reg', "Images"]]

def add_watermark_to_image(image_path, text="Sourced from Leominstercarauctions"):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)

        try:
            font = ImageFont.truetype("arial.ttf", 50)
        except:
            font = ImageFont.load_default()

        margin = 10
        bbox = draw.textbbox((0, 0), text, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        x, y = image.width - tw - margin, image.height - th - margin

        draw.rectangle([x - margin, y - margin, x + tw + margin, y + th + margin],
                       fill=(0,0,0,160))

        draw.text((x, y), text, font=font, fill=(255,255,255,200))

        watermarked = Image.alpha_composite(image, txt_layer).convert("RGB")
        watermarked.save(image_path)

        print(f"✔ Watermarked: {image_path}")

    except Exception as e:
        print(f"⚠ Watermark Error: {e}")

def download_images(data, main_folder="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for index, row in data.iterrows():
        reg_no = str(row["Reg"]).strip()
        if not reg_no:
            continue

        img_urls = [u for u in re.split(r',\s*', str(row["Images"])) if u]
        if not img_urls:
            continue

        reg_folder = os.path.join(main_folder, reg_no)
        os.makedirs(reg_folder, exist_ok=True)

        def save_img(url, folder, idx):
            url = url.strip()
            if not url:
                return

            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            parsed = urlparse(url)
            if not parsed.netloc:
                print(f"❌ Invalid URL Skipped: {url}")
                return

            full_path = os.path.join(folder, f"{reg_no}_{idx}.jpg")

            if os.path.exists(full_path):
                print(f"⏩ Skipped (Exists): {full_path}")
                return

            try:
                response = requests.get(url, stream=True, timeout=20)
                response.raise_for_status()

                with open(full_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(full_path)
                print(f"📌 Saved: {full_path}")

            except Exception as e:
                print(f"⚠ Error downloading: {url} -> {e}")

        for i, url in enumerate(img_urls):
            save_img(url, reg_folder, i+1)

def start_funcs():
    t1 = threading.Thread(target=download_images, args=(reg_img,))
    t1.start()
    t1.join()

if __name__ == "__main__":
    start_funcs()


✔ Watermarked: Images\AF06MYX\AF06MYX_1.jpg
📌 Saved: Images\AF06MYX\AF06MYX_1.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_2.jpg
📌 Saved: Images\AF06MYX\AF06MYX_2.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_3.jpg
📌 Saved: Images\AF06MYX\AF06MYX_3.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_4.jpg
📌 Saved: Images\AF06MYX\AF06MYX_4.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_5.jpg
📌 Saved: Images\AF06MYX\AF06MYX_5.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_6.jpg
📌 Saved: Images\AF06MYX\AF06MYX_6.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_7.jpg
📌 Saved: Images\AF06MYX\AF06MYX_7.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_8.jpg
📌 Saved: Images\AF06MYX\AF06MYX_8.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_9.jpg
📌 Saved: Images\AF06MYX\AF06MYX_9.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_10.jpg
📌 Saved: Images\AF06MYX\AF06MYX_10.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_11.jpg
📌 Saved: Images\AF06MYX\AF06MYX_11.jpg
✔ Watermarked: Images\AF06MYX\AF06MYX_12.jpg
📌 Saved: Images\AF06MYX\AF06MYX_12.jpg
✔ Watermar